## Data Quality
This script checks the quality of the merged.csv dataset based on the following indicators:
- currency
- completeness
- consistency

It is worth noting that no previous cleaning or processing has been made on the data yet except for a merge between the two datasets (RISE_dataset and the global power plant one).


### Currency
To measure how up to date the dataset is we'll consider a few key checks, and since the file doesn’t have a date column it is not possible to directly determine how current the data is just from the file content. We however know that the RISE dataset is from 2016 from the following link https://energydata.info/dataset/world-regulatory-indicators-sustainable-energy-2016 as it was last updated on September 30, 2024, while the global power plant dataset has been last updated on October 16, 2024 (link for the dataset: https://datasets.wri.org/datasets/global-power-plant-database?map=eyJ2aWV3U3RhdGUiOnsibGF0aXR1ZGUiOjAsImxvbmdpdHVkZSI6MCwiem9vbSI6MywiYmVhcmluZyI6MCwicGl0Y2giOjAsInBhZGRpbmciOnsidG9wIjowLCJib3R0b20iOjAsImxlZnQiOjAsInJpZ2h0IjowfX0sImJhc2VtYXAiOiJsaWdodCIsImJvdW5kYXJpZXMiOmZhbHNlLCJsYWJlbHMiOiJkYXJrIiwiYWN0aXZlTGF5ZXJHcm91cHMiOlt7ImRhdGFzZXRJZCI6IjUzNjIzZGZkLTNkZjYtNGYxNS1hMDkxLTY3NDU3Y2RiNTcxZiIsImxheWVycyI6WyIyYTY5NDI4OS1mZWM5LTRiZmUtYTZkMi01NmMzODY0ZWMzNDkiXX1dLCJib3VuZHMiOnsiYmJveCI6bnVsbCwib3B0aW9ucyI6e319LCJsYXllcnNQYXJzZWQiOltbIjJhNjk0Mjg5LWZlYzktNGJmZS1hNmQyLTU2YzM4NjRlYzM0OSIseyJ2aXNpYmlsaXR5Ijp0cnVlLCJhY3RpdmUiOnRydWUsIm9wYWNpdHkiOjEsInpJbmRleCI6MTF9XV19).


Both datasets have been last updated in 2024, meaning that the currency of the merged dataset can be considered strong since the data is representative of the same general time period and it reflects the current state of renewable energy systems (assuming it’s now mid-2025). The dataset is suitable for current decision-making, forecasting, reporting, and policy analysis although it is not perfectly synchronised, but still decent for time-sensitive use cases and for tasks like investment planning, national comparisons, or climate reporting.
 

## Completeness
Completeness refers to:
- Coverage: Are all relevant entities included?
- Attribute completeness: Are all the data fields filled in for each row?

About coverage, there are ~195 recognized countries in the world: 193 UN members + 2 observers (Vatican, Palestine) so this means that it is partial: the merged dataset includes roughly half of the world’s countries, however it is still regionally representative as the countries listed span all continents and economically significant as it includes major energy consumers (U.S., China, EU countries, India, Brazil).



To check for attribute completeness we check for null values:

In [3]:
import pandas as pd

df = pd.read_csv("../Datasets_and_maps/merged.csv")
missing_summary = df.isnull().sum()
print(missing_summary)


Unnamed: 0                0
Country                   0
Region                    0
Income group              0
Renewable Energy score    0
Total_Capacity_MW         0
Renewable_Capacity_MW     0
Renewable_Share           0
dtype: int64


From the results no data is missing, however if in the Renewable_Capacity_MW column there is a 0 this means the country in question does not use any renewable energy, hence the 0.

## Consistency
Consistency means ensuring that values across related columns logically align and conform to expected relationships or rules. We will check the following:

In [13]:
# No Capacity Should Be Negative:
print((df[['Total_Capacity_MW', 'Renewable_Capacity_MW', 'Renewable_Share']] < 0).any())

# Renewable Share Between 0 and 1
df[(df['Renewable_Share'] < 0) | (df['Renewable_Share'] > 1)]

# Income Group Values Are Uniform
print(df['Income group'].value_counts())

# Check for Duplicated Entries
print(df['Country'].duplicated().sum())

# Is the Renewable Energy score consistent with the actual share? If it is 0 then renewable share must also be zero
# Filter rows where score is 0 but share is not
violations = df[(df['Renewable Energy score'] == 0) & (df['Renewable_Share'] != 0)]

print(f"Rows violating the rule: {len(violations)}")
display(violations[['Country', 'Renewable Energy score', 'Renewable_Share']])





Total_Capacity_MW        False
Renewable_Capacity_MW    False
Renewable_Share          False
dtype: bool
Income group
High income            25
Lower middle income    25
Low income             22
Upper middle income    20
Name: count, dtype: int64
0
Rows violating the rule: 0


,Country,Renewable Energy score,Renewable_Share
